In [6]:
import numpy as np
import pandas as pd
import sklearn
import pyod

In [7]:
acad = pd.read_csv("energy_dataset/acad_build_mains.csv")
lib = pd.read_csv("energy_dataset/library_build_mains.csv")

In [8]:
# convert timestamp and set as index
acad['timestamp'] = pd.to_datetime(acad['timestamp'], unit='s')
acad = acad.set_index('timestamp')

lib['timestamp'] = pd.to_datetime(lib['timestamp'], unit='s')
lib = lib.set_index('timestamp')

In [9]:
# resampling to 15 min intervals, fwd fill gaps
acad = acad.resample('15min').mean().ffill()
lib = lib.resample('15min').mean().ffill()


print("Acad shape:", acad.shape)
print("Lib shape:", lib.shape)
print("\nAcad date range:", acad.index.min(), "to", acad.index.max())
print("Lib date range:", lib.index.min(), "to", lib.index.max())
print("\nNull values in acad:\n", acad.isnull().sum())
print("\nNull values in lib:\n", lib.isnull().sum())

Acad shape: (154080, 5)
Lib shape: (154080, 5)

Acad date range: 2013-08-09 18:30:00 to 2017-12-31 18:15:00
Lib date range: 2013-08-09 18:30:00 to 2017-12-31 18:15:00

Null values in acad:
 power               0
current         14118
voltage             0
frequency           0
power_factor        0
dtype: int64

Null values in lib:
 power               0
current         14118
voltage             0
frequency           0
power_factor        0
dtype: int64


In [10]:
# 6 month window = Jan 2016 to June 2016
acad_6m = acad['2016-01-01':'2016-06-30'][['power']]
lib_6m = lib['2016-01-01':'2016-06-30'][['power']]

In [11]:
# power col normalisation
from sklearn.preprocessing import MinMaxScaler
scaler_acad = MinMaxScaler() # PyOD algorithms are sensitive to scale 
scaler_lib = MinMaxScaler()

acad_6m['power_norm'] = scaler_acad.fit_transform(acad_6m[['power']])
lib_6m['power_norm'] = scaler_lib.fit_transform(lib_6m[['power']])


print("Acad 6m shape:", acad_6m.shape)
print("Lib 6m shape:", lib_6m.shape)
print("\nAcad power stats:")
print(acad_6m['power'].describe())
print("\nLib power stats:")
print(lib_6m['power'].describe())

Acad 6m shape: (17472, 2)
Lib 6m shape: (17472, 2)

Acad power stats:
count    17472.000000
mean     28329.546958
std      11842.915272
min       9523.612029
25%      20474.560996
50%      24478.141602
75%      32145.937093
max      68924.378255
Name: power, dtype: float64

Lib power stats:
count    17472.000000
mean     10928.896216
std       8288.338819
min        771.375872
25%       4623.270401
50%       8132.746110
75%      15920.945841
max      42349.391732
Name: power, dtype: float64


In [12]:
# scenario 1 : point outliers

import random

def inject_point_outliers(df, n=50, multiplier=3.0, seed=42):
    data = df.copy()
    data['label'] = 0  
    
    random.seed(seed)
    outlier_indices = random.sample(range(len(data)), n)

    data.iloc[outlier_indices, data.columns.get_loc('power_norm')] *= multiplier 
    
    data['power_norm'] = data['power_norm'].clip(upper=1.0)
    
    data.iloc[outlier_indices, data.columns.get_loc('label')] = 1
    
    return data

acad_s1 = inject_point_outliers(acad_6m)
lib_s1 = inject_point_outliers(lib_6m)

acad_s1.to_csv('acad_scenario1.csv')
lib_s1.to_csv('lib_scenario1.csv')

print("Acad scenario 1 — anomaly count:", acad_s1['label'].sum())
print("Lib scenario 1 — anomaly count:", lib_s1['label'].sum())
print("\nSample anomalous rows in acad:")
print(acad_s1[acad_s1['label'] == 1].head())

Acad scenario 1 — anomaly count: 50
Lib scenario 1 — anomaly count: 50

Sample anomalous rows in acad:
                            power  power_norm  label
timestamp                                           
2016-01-03 05:00:00  12928.429721    0.171958      1
2016-01-09 12:45:00  25780.487370    0.821044      1
2016-01-10 01:15:00  15228.081217    0.288101      1
2016-01-11 04:00:00  46791.883855    1.000000      1
2016-01-11 20:15:00  19342.303711    0.495887      1


In [13]:
# scenario 2 : power spike (sustained surge lasting 2 hours (8 readings at 15min intervals)

def inject_power_spike(df, start_idx=7000, duration=8, multiplier=2.5, seed=42):
    data = df.copy()
    data['label'] = 0
    
    # inject a sustained spike for 'duration' consecutive readings
    data.iloc[start_idx:start_idx+duration, data.columns.get_loc('power_norm')] *= multiplier
    data['power_norm'] = data['power_norm'].clip(upper=1.0)
    
    data.iloc[start_idx:start_idx+duration, data.columns.get_loc('label')] = 1  
    return data

acad_s2 = inject_power_spike(acad_6m, start_idx=329)
lib_s2 = inject_power_spike(lib_6m, start_idx=329)

acad_s2.to_csv('acad_scenario2.csv')
lib_s2.to_csv('lib_scenario2.csv')

print("Spike window in acad:")
print(acad_s2[acad_s2['label'] == 1][['power', 'power_norm', 'label']])

Spike window in acad:
                            power  power_norm  label
timestamp                                           
2016-01-04 10:15:00  35592.614454    1.000000      1
2016-01-04 10:30:00  36253.890105    1.000000      1
2016-01-04 10:45:00  35661.332422    1.000000      1
2016-01-04 11:00:00  34894.233463    1.000000      1
2016-01-04 11:15:00  31885.733008    0.941155      1
2016-01-04 11:30:00  32505.441211    0.967236      1
2016-01-04 11:45:00  35038.926107    1.000000      1
2016-01-04 12:00:00  34208.737175    1.000000      1


In [14]:
# scenario 3 : trend drift (gradual creep — power slowly increases over 2 weeks beyond what's normal)

# We take a 2 week window (2 weeks × 7 days × 24 hours × 4 readings = 1344 rows) and multiply each 
# reading by a gradually increasing factor — starting at 1.0 and ending at 1.5. So by the end of 2
# weeks power is 50% higher than normal.

def inject_trend_drift(df, start_idx=1000, duration=1344, max_multiplier=1.5):
    data = df.copy()
    data['label'] = 0
    
    # gradually increasing multiplier from 1.0 to max_multiplier
    multipliers = np.linspace(1.0, max_multiplier, duration)
    
    for i, mult in enumerate(multipliers):
        idx = start_idx + i
        if idx >= len(data):
            break
        data.iloc[idx, data.columns.get_loc('power_norm')] *= mult
    
    data['power_norm'] = data['power_norm'].clip(upper=1.0)
    
    # label the whole drift window as anomalous
    data.iloc[start_idx:start_idx+duration, 
              data.columns.get_loc('label')] = 1
    
    return data

acad_s3 = inject_trend_drift(acad_6m, start_idx=1000)
lib_s3 = inject_trend_drift(lib_6m, start_idx=1000)

acad_s3.to_csv('acad_scenario3.csv')
lib_s3.to_csv('lib_scenario3.csv')

print("Drift window — first and last 3 rows in acad:")
print(acad_s3[acad_s3['label'] == 1][['power', 'power_norm']].head(3))
print("...")
print(acad_s3[acad_s3['label'] == 1][['power', 'power_norm']].tail(3))
print("\nTotal anomalous rows:", acad_s3['label'].sum())

Drift window — first and last 3 rows in acad:
                            power  power_norm
timestamp                                    
2016-01-11 10:00:00  34516.789583    0.420755
2016-01-11 10:15:00  35506.540495    0.437580
2016-01-11 10:30:00  34152.817903    0.414936
...
                            power  power_norm
timestamp                                    
2016-01-25 09:15:00  51729.330729    1.000000
2016-01-25 09:30:00  41330.798959    0.803002
2016-01-25 09:45:00  39674.078516    0.761366

Total anomalous rows: 1344


In [15]:
# scenario 4 : contextual anomaly

def inject_contextual_anomaly(df, target_hour=3, n_days=10, multiplier=2.0):
    data = df.copy()
    data['label'] = 0
    
    # find all 3am readings
    night_mask = data.index.hour == target_hour
    night_indices = np.where(night_mask)[0]
    
    # pick first n_days worth of 3am readings (4 per day at 15min intervals)
    target_indices = night_indices[:n_days * 4]
    
    # multiply those readings to simulate high consumption at night
    for idx in target_indices:
        data.iloc[idx, data.columns.get_loc('power_norm')] *= multiplier
    
    data['power_norm'] = data['power_norm'].clip(upper=1.0)
    data.iloc[target_indices, data.columns.get_loc('label')] = 1
    
    return data

acad_s4 = inject_contextual_anomaly(acad_6m)
lib_s4 = inject_contextual_anomaly(lib_6m)

acad_s4.to_csv('acad_scenario4.csv')
lib_s4.to_csv('lib_scenario4.csv')

print("Contextual anomaly rows in acad:")
print(acad_s4[acad_s4['label'] == 1][['power', 'power_norm', 'label']].head(8))
print("\nTotal anomalous rows:", acad_s4['label'].sum())

Contextual anomaly rows in acad:
                            power  power_norm  label
timestamp                                           
2016-01-01 03:00:00  16872.226563    0.247425      1
2016-01-01 03:15:00  17909.597070    0.282353      1
2016-01-01 03:30:00  28513.621029    0.639386      1
2016-01-01 03:45:00  34941.341146    0.855805      1
2016-01-02 03:00:00  15717.754297    0.208554      1
2016-01-02 03:15:00  17978.955144    0.284688      1
2016-01-02 03:30:00  18263.786425    0.294278      1
2016-01-02 03:45:00  19462.267513    0.334631      1

Total anomalous rows: 40


In [16]:
# scenario 5 : anomalous subsequence (flat/zero readings mid-day)

def inject_flat_subsequence(df, target_hour=12, duration=8, flat_value=0.0):
    data = df.copy()
    data['label'] = 0
    
    # find a midday window (12pm readings)
    midday_mask = (data.index.hour == target_hour) & (data.index.minute == 0)
    midday_indices = np.where(midday_mask)[0]
    
    # pick the first occurrence and take 'duration' readings from there (2 hours = 8 x 15min)
    start = midday_indices[0]
    end = start + duration
    
    data.iloc[start:end, data.columns.get_loc('power_norm')] = flat_value
    data.iloc[start:end, data.columns.get_loc('label')] = 1
    
    return data

acad_s5 = inject_flat_subsequence(acad_6m)
lib_s5 = inject_flat_subsequence(lib_6m)
acad_s5.to_csv('acad_scenario5.csv')
lib_s5.to_csv('lib_scenario5.csv')

print("Flat subsequence in acad:")
print(acad_s5[acad_s5['label'] == 1][['power', 'power_norm', 'label']])
print("\nTotal anomalous rows:", acad_s5['label'].sum())

Flat subsequence in acad:
                            power  power_norm  label
timestamp                                           
2016-01-01 12:00:00  24648.922069         0.0      1
2016-01-01 12:15:00  24292.405469         0.0      1
2016-01-01 12:30:00  23021.034180         0.0      1
2016-01-01 12:45:00  22094.481445         0.0      1
2016-01-01 13:00:00  21453.965951         0.0      1
2016-01-01 13:15:00  20750.333724         0.0      1
2016-01-01 13:30:00  21382.327149         0.0      1
2016-01-01 13:45:00  22805.697851         0.0      1

Total anomalous rows: 8


In [17]:
# scenario 6 : spatial disagreement (acad and lib diverge at same timestamps)

def inject_spatial_disagreement(df_acad, df_lib, start_idx=500, duration=32, multiplier=3.0):
    acad = df_acad.copy()
    lib = df_lib.copy()
    acad['label'] = 0
    lib['label'] = 0
    
    # spike acad power up, suppress lib power down at the same window
    acad.iloc[start_idx:start_idx+duration, acad.columns.get_loc('power_norm')] *= multiplier
    lib.iloc[start_idx:start_idx+duration, lib.columns.get_loc('power_norm')] *= 0.2
    
    acad['power_norm'] = acad['power_norm'].clip(upper=1.0)
    lib['power_norm'] = lib['power_norm'].clip(lower=0.0)
    
    # label the divergence window in both
    acad.iloc[start_idx:start_idx+duration, acad.columns.get_loc('label')] = 1
    lib.iloc[start_idx:start_idx+duration, lib.columns.get_loc('label')] = 1
    
    return acad, lib

acad_s6, lib_s6 = inject_spatial_disagreement(acad_6m, lib_6m)
acad_s6.to_csv('acad_scenario6.csv')
lib_s6.to_csv('lib_scenario6.csv')

print("Spatial disagreement window — acad:")
print(acad_s6[acad_s6['label'] == 1][['power', 'power_norm', 'label']].head(5))
print("\nSpatial disagreement window — lib:")
print(lib_s6[lib_s6['label'] == 1][['power', 'power_norm', 'label']].head(5))
print("\nAnomaly count acad:", acad_s6['label'].sum(), "| lib:", lib_s6['label'].sum())

Spatial disagreement window — acad:
                            power  power_norm  label
timestamp                                           
2016-01-06 05:00:00  49642.011720         1.0      1
2016-01-06 05:15:00  50614.800260         1.0      1
2016-01-06 05:30:00  50896.736979         1.0      1
2016-01-06 05:45:00  50491.344271         1.0      1
2016-01-06 06:00:00  51994.607811         1.0      1

Spatial disagreement window — lib:
                            power  power_norm  label
timestamp                                           
2016-01-06 05:00:00  15825.667593    0.072415      1
2016-01-06 05:15:00  11402.224611    0.051137      1
2016-01-06 05:30:00  16311.729003    0.074753      1
2016-01-06 05:45:00  13696.079379    0.062171      1
2016-01-06 06:00:00  11090.447509    0.049637      1

Anomaly count acad: 32 | lib: 32


In [18]:
# running pyOD on all scenarios
# for each scenario CSV, load it, use power_norm as the feature, run all 10 algorithms,
# and collect precision, recall, F1, and AUC-ROC.

from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from pyod.models.knn import KNN
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.models.suod import SUOD
from pyod.models.cblof import CBLOF
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 10 algorithms
algorithms = {
    'IForest': IForest(contamination=0.05, random_state=42),
    'LOF': LOF(contamination=0.05),
    'OCSVM': OCSVM(contamination=0.05),
    'KNN': KNN(contamination=0.05),
    'HBOS': HBOS(contamination=0.05),
    'COPOD': COPOD(contamination=0.05),
    'ECOD': ECOD(contamination=0.05),
    'SUOD': SUOD(contamination=0.05),
    'CBLOF': CBLOF(contamination=0.05, random_state=42),
    'xStream': IForest(contamination=0.05, random_state=0),  
}

scenario_files = {
    'acad_s1': 'acad_scenario1.csv', 'lib_s1': 'lib_scenario1.csv',
    'acad_s2': 'acad_scenario2.csv', 'lib_s2': 'lib_scenario2.csv',
    'acad_s3': 'acad_scenario3.csv', 'lib_s3': 'lib_scenario3.csv',
    'acad_s4': 'acad_scenario4.csv', 'lib_s4': 'lib_scenario4.csv',
    'acad_s5': 'acad_scenario5.csv', 'lib_s5': 'lib_scenario5.csv',
    'acad_s6': 'acad_scenario6.csv', 'lib_s6': 'lib_scenario6.csv',
}

results = []

for scenario_name, filepath in scenario_files.items():
    df = pd.read_csv(filepath, index_col='timestamp', parse_dates=True)
    X = df[['power_norm']].values
    y_true = df['label'].values

    for algo_name, model in algorithms.items():
        # re-instantiating to avoid state leaking between scenarios
        model = algorithms[algo_name].__class__(**algorithms[algo_name].get_params())
        
        try:
            model.fit(X)
            y_pred = model.labels_
            y_scores = model.decision_scores_

            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            auc = roc_auc_score(y_true, y_scores)

            results.append({
                'scenario': scenario_name,
                'algorithm': algo_name,
                'precision': round(precision, 3),
                'recall': round(recall, 3),
                'f1': round(f1, 3),
                'auc_roc': round(auc, 3),
            })
        except Exception as e:
            print(f"Failed — {algo_name} on {scenario_name}: {e}")
            results.append({
                'scenario': scenario_name,
                'algorithm': algo_name,
                'precision': None, 'recall': None, 'f1': None, 'auc_roc': None,
            })

results_df = pd.DataFrame(results)
results_df.to_csv('pyod_results.csv', index=False)
print(results_df.to_string())

RandomForestRegressor()



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.8s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.2s finished


RandomForestRegressor()



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.7s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.7s finished


RandomForestRegressor()



...

RandomForestRegressor()



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.7s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.3s finished


RandomForestRegressor()



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.7s finished
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.5s finished


    scenario algorithm  precision  recall     f1  auc_roc
0    acad_s1   IForest      0.021   0.360  0.039    0.849
1    acad_s1       LOF      0.018   0.320  0.035    0.615
2    acad_s1     OCSVM      0.021   0.360  0.039    0.794
3    acad_s1       KNN      0.005   0.080  0.009    0.552
4    acad_s1      HBOS      0.035   0.360  0.063    0.863
5    acad_s1     COPOD      0.021   0.360  0.039    0.838
6    acad_s1      ECOD      0.018   0.320  0.035    0.769
7    acad_s1      SUOD      0.021   0.360  0.039    0.847
8    acad_s1     CBLOF      0.026   0.460  0.050    0.835
9    acad_s1   xStream      0.021   0.360  0.039    0.845
10    lib_s1   IForest      0.032   0.560  0.061    0.822
11    lib_s1       LOF      0.003   0.060  0.006    0.503
12    lib_s1     OCSVM      0.032   0.560  0.061    0.774
13    lib_s1       KNN      0.014   0.240  0.026    0.442
14    lib_s1      HBOS      0.038   0.600  0.071    0.839
15    lib_s1     COPOD      0.032   0.560  0.061    0.822
16    lib_s1  

In [19]:
# Result Table
# pivot table - rows = algorithms, columns = scenarios, values = F1
pivot_f1 = results_df.pivot(index='algorithm', columns='scenario', values='f1')
pivot_auc = results_df.pivot(index='algorithm', columns='scenario', values='auc_roc')

print("=== F1 SCORES ===")
print(pivot_f1.to_string())
print("\n=== AUC-ROC SCORES ===")
print(pivot_auc.to_string())

# save both
pivot_f1.to_csv('results_f1.csv')
pivot_auc.to_csv('results_auc.csv')

=== F1 SCORES ===
scenario   acad_s1  acad_s2  acad_s3  acad_s4  acad_s5  acad_s6  lib_s1  lib_s2  lib_s3  lib_s4  lib_s5  lib_s6
algorithm                                                                                                      
CBLOF        0.050    0.018    0.113    0.028    0.000    0.071   0.061   0.009   0.323   0.002   0.000     0.0
COPOD        0.039    0.018    0.115    0.028    0.018    0.071   0.061   0.007   0.307   0.002   0.018     0.0
ECOD         0.035    0.018    0.124    0.028    0.018    0.071   0.058   0.005   0.198   0.000   0.018     0.0
HBOS         0.063    0.031    0.114    0.047    0.000    0.113   0.071   0.010   0.272   0.002   0.000     0.0
IForest      0.039    0.018    0.117    0.028    0.018    0.071   0.061   0.005   0.303   0.002   0.018     0.0
KNN          0.009    0.005    0.100    0.011    0.000    0.007   0.026   0.009   0.298   0.007   0.000     0.0
LOF          0.035    0.016    0.083    0.031    0.018    0.002   0.006   0.005   0.07